<a href="https://colab.research.google.com/github/lucianoselimaj/SigExt/blob/second_extension_dir_management/extensions/02_three_layer_agentic_refinement/semantic_evaluator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 1. Environment Setup

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# @title 2. Utility Functions for Data Ingestion
# @markdown Defines helper functions to read standard JSON files and line by line JSONL files into Python dictionaries and lists.

import json
def read_jsonl_file(path):
  data = []
  with open(path, 'r', encoding='utf-8') as f:
    for line in f:
      data.append(json.loads(line.strip()))
  return data

def read_json_file(path):
  with open(path, 'r', encoding='utf-8') as f:
    data = json.load(f)
  return data


In [ ]:
# @title 3. Configuration & File Paths
# @markdown Specifies the storage paths for the baseline predictions (Experiment 1), extended model predictions (Experiment 2), and the ground-truth evaluation dataset.

experiment_1_path = "/content/drive/MyDrive/DNLP-storage/SigExt/experiments/cnn_extsig_predictions_mistral-small-2603_k15/test_predictions.json" #@param{type:"string"}
experiment_2_path = "/content/drive/MyDrive/DNLP-storage/SigExt/experiments/cnn_extsig_predictions_mistral-small-2603_k15_extention2/test_predictions.json" #@param{type:"string"}
dataset_path = "/content/drive/MyDrive/DNLP-storage/SigExt/experiments/cnn_extsig_predictions_mistral-small-2603_k15/test_dataset.jsonl"

In [ ]:
# @title 4. Data Loading & Inspection
# @markdown Loads the text generation outputs from both experiments alongside the reference dataset, printing the first sample of each to verify structural integrity.
import json


experiment_1_data = read_json_file(experiment_1_path)
experiment_2_data = read_json_file(experiment_2_path) # Assuming experiment_2_path is also a standard .json file
dataset = read_jsonl_file(dataset_path)

print(experiment_1_data[0])
print(experiment_2_data[0])
print(dataset[0]['raw_output'])

A former Japanese school principal, Yuhei Takashima, is facing criminal charges in Tokyo after police arrested him for allegedly paying for sex with over 12,000 women—some as young as 14—in the Philippines, where he also photographed obscene acts and produced pornography. Police seized 147,600 photos documenting his activities, with investigations revealing he traveled to Manila repeatedly since 1988 to "buy sex," citing its affordability.
A former school principal in Japan, Yuhei Takashima, is facing criminal charges after police arrested him for allegedly paying for sex with over 12,000 women, including minors as young as 14, during frequent trips to the Philippines over 25 years. Police seized 147,600 photos documenting his activities in Manila, with investigations revealing he continued to "buy sex" due to its low cost, while authorities in Tokyo and Yokohama condemned the "very regrettable" crime involving minors, and said Takashima's case had been under investigation since 2013.


In [ ]:
# @title 5. Semantic Similarity Engine
# @markdown Initializes a pre-trained Sentence-BERT model to encode text sequences into dense vector embeddings and compute their directional cosine similarity.
from sentence_transformers import SentenceTransformer, util


sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

def compare_with_sbert(reference, sentence_A, sentence_B):

    embeddings = sbert_model.encode([reference, sentence_A, sentence_B])


    vec_ref = embeddings[0]
    vec_A = embeddings[1]
    vec_B = embeddings[2]

    sim_A = util.cos_sim(vec_ref, vec_A).item()
    sim_B = util.cos_sim(vec_ref, vec_B).item()

    return sim_A, sim_B



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# @title 6. Evaluation Metric Mapping
# @markdown Configures whether the generated outputs should be evaluated against the human-written target label (`raw_output`) or the source context (`raw_input`).

compare_with_target_lable = True #@param{type:"boolean"}
ref = "raw_output" if  compare_with_target_lable else "raw_input"

In [ ]:

# @title 7. Comparative Evaluation Loop
# @markdown Iterates through the dataset to calculate semantic similarity scores for both configurations, generating a comprehensive descriptive report to measure performance gains.

import numpy as np

s1 =experiment_1_data[0]
s2 = experiment_2_data[0]
reference = dataset[0][ref]
dataset_size = len(reference)
scores = np.zeros((dataset_size, 3))
sim_a, sim_b = compare_with_sbert(reference=reference, sentence_A=s1,sentence_B = s2)

for i in range(dataset_size):
  try:
    s1 =experiment_1_data[i]
    s2 = experiment_2_data[i]
    reference = dataset[i][ref]
    sim_a, sim_b = compare_with_sbert(reference=reference, sentence_A=s1,sentence_B = s2)
    scores[i, 0] = sim_a
    scores[i, 1] = sim_b
    scores[i, 2] = sim_b - sim_a

  except:
    pass

print(np.mean(scores, axis=0))

[0.75227327 0.75348534 0.00121207]
